# 01. Khám phá dữ liệu (EDA)
Đề tài 2: dự đoán nguy cơ bệnh tim mạch. Bộ dữ liệu Heart Disease (Cleveland, UCI), 303 dòng, 13 đặc trưng.

In [1]:
import sys; sys.path.append('..')
from src.common import *
import warnings; warnings.filterwarnings('ignore')

## 1. Nguồn dữ liệu
Bản CSV 303 dòng của bộ Cleveland (UCI Machine Learning Repository: Janosi, Steinbrunn, Pfisterer, Detrano) được tải từ kho GitHub `kb22/Heart-Disease-Prediction`. **Cần ghi nguồn gốc UCI và kiểm tra giấy phép trước khi nộp** (xem mục Hạn chế trong báo cáo).

In [2]:
df = pd.read_csv(DATA / 'heart_raw.csv')
print('shape:', df.shape)
df.head()

shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 2. Kiểu dữ liệu, thống kê mô tả

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.37,9.08,29.0,47.5,55.0,61.0,77.0
sex,303.0,0.68,0.47,0.0,0.0,1.0,1.0,1.0
cp,303.0,0.97,1.03,0.0,0.0,1.0,2.0,3.0
trestbps,303.0,131.62,17.54,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.26,51.83,126.0,211.0,240.0,274.5,564.0
fbs,303.0,0.15,0.36,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.53,0.53,0.0,0.0,1.0,1.0,2.0
thalach,303.0,149.65,22.91,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.33,0.47,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.04,1.16,0.0,0.0,0.8,1.6,6.2


## 3. Giá trị thiếu và trùng lặp

In [5]:
print('Giá trị thiếu theo cột:'); print(df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else 'Không có NaN')
print('Số dòng trùng lặp:', df.duplicated().sum())
df[df.duplicated(keep=False)]

Giá trị thiếu theo cột:
Không có NaN
Số dòng trùng lặp: 1


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
163,38,1,2,138,175,0,1,173,0,0.0,2,4,2,1
164,38,1,2,138,175,0,1,173,0,0.0,2,4,2,1


## 4. Giá trị không hợp lệ (mã sai so với mô tả cột)
Theo mô tả: `ca` nhận 0 đến 3, `thal` nhận 1 đến 3. Các giá trị vi phạm sẽ được xử lý ở notebook 02.

In [6]:
print('ca:', df['ca'].value_counts().sort_index().to_dict())
print('thal:', df['thal'].value_counts().sort_index().to_dict())
print('Số dòng ca=4:', (df.ca == 4).sum(), '| số dòng thal=0:', (df.thal == 0).sum())

ca: {0: 175, 1: 65, 2: 38, 3: 20, 4: 5}
thal: {0: 2, 1: 18, 2: 166, 3: 117}
Số dòng ca=4: 5 | số dòng thal=0: 2


## 5. Kiểm tra ý nghĩa nhãn `target`
Trong bản CSV này, `target = 1` đi kèm nhịp tim tối đa **cao hơn**, oldpeak **thấp hơn**, ít đau thắt ngực khi gắng sức và ít mạch hẹp hơn. Về mặt y học, đây là hồ sơ của nhóm **không bệnh**. Nhiều bản sao trên mạng của bộ dữ liệu này có nhãn bị đảo so với bản gốc UCI (UCI gán 0 = không bệnh), nên cần kiểm chứng bằng dữ liệu thay vì tin tên cột.

In [7]:
chk = df.groupby('target')[['age','thalach','oldpeak','exang','ca']].mean().round(2)
chk

,age,thalach,oldpeak,exang,ca
target,,,,,
0,56.6,139.10,1.59,0.55,1.17
1,52.5,158.47,0.58,0.14,0.36


**Kết luận:** dùng nhãn mới `HeartDisease = 1 - target` (1 = có bệnh). Quyết định này được ghi lại trong báo cáo và là điểm nên chuẩn bị sẵn khi vấn đáp.

In [8]:
print(df['target'].value_counts().rename({0: 'target=0', 1: 'target=1'}))

target
target=1    165
target=0    138
Name: count, dtype: int64


## 6. Bảng mô tả cột

In [9]:
pd.read_csv(DATA / 'data_dictionary.csv')

,Cột,Ý nghĩa,Kiểu,Giá trị / đơn vị
0,age,Tuổi,Số nguyên,năm
1,sex,Giới tính,Nhị phân,0 = nữ; 1 = nam
2,cp,Loại đau ngực,Phân loại,0-3 (mã hóa theo bộ dữ liệu)
3,trestbps,Huyết áp tâm thu lúc nghỉ,Số nguyên,mmHg
4,chol,Cholesterol huyết thanh,Số nguyên,mg/dl
5,fbs,Đường huyết đói > 120 mg/dl,Nhị phân,0 = không; 1 = có
6,restecg,Kết quả điện tâm đồ lúc nghỉ,Phân loại,0-2
7,thalach,Nhịp tim tối đa đạt được,Số nguyên,nhịp/phút
8,exang,Đau thắt ngực khi gắng sức,Nhị phân,0 = không; 1 = có
9,oldpeak,ST chênh xuống do gắng sức so với lúc nghỉ,Số thực,đơn vị ST
